<a href="https://colab.research.google.com/github/nmquintero/MLandDS/blob/main/optimization/Optimization_Tasks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Optimization. Practical Tasks

Please, execute two rows of code below at first.

In [1]:
!pip install git+https://github.com/mehalyna/cooltest.git

  Cloning https://github.com/mehalyna/cooltest.git to /tmp/pip-req-build-_xrfwq7q
  Running command git clone --filter=blob:none --quiet https://github.com/mehalyna/cooltest.git /tmp/pip-req-build-_xrfwq7q
  Resolved https://github.com/mehalyna/cooltest.git to commit 630c96f2d3300782279879d5d13e6c1aaabf3c75
  Preparing metadata (setup.py) ... done
  Created wheel for cooltest: filename=cooltest-26.22-py3-none-any.whl size=5447 sha256=d3969c6c1ed1fb61d1bd5da5a7ab5caf5e1471571a217e067d0d5bc847251151
  Stored in directory: /tmp/pip-ephem-wheel-cache-hkzkg2ku/wheels/74/9f/b6/427a02d44dc13bccb5245586783e7de2f1efb9556847d744da
Successfully built cooltest


In [2]:
from cooltest.test_cool_3 import *

Pass


# Task 1. Resource Scheduling

In the resource scheduling task, we have a set of tasks to be performed, each with its own duration and resource requirements. Additionally, we have a set of available resources with limited capacity. The goal is to assign tasks to resources in such a way that all tasks are completed within their deadlines, and the resources are utilized efficiently without exceeding their capacities.

Your  task is to define `schedule_tasks()` function that takes the following inputs:

- `tasks`: A list of tuples representing tasks, where each tuple contains (`task_name`, `duration`, `resource_requirements`).
- `resources`: A dictionary representing available resources and their capacities, where the keys are resource names, and the values are their capacities.
- `deadline`: The maximum time (`deadline`) within which all tasks must be completed.

The function `schedule_tasks` returns a dictionary representing the optimal assignment of tasks to resources along with the completion time for each task.

>**Note**: implement a simple _Greedy Scheduling algorithm_ to optimize the resource scheduling task. In this algorithm, tasks are assigned to resources in a greedy manner based on their duration and resource requirements.

In [4]:
"""
  El enfoque Greedy busca la opción más inmediata y beneficiosa en cada paso.
  Ordenamos las tareas por su duración de menor a mayor. Luego, comprobamos si hay suficientes
  recursos disponibles en nuestro diccionario general de recursos. Si los recursos lo permiten,
  asignamos la tarea, reducimos las capacidades en resources y anotamos el tiempo de duración; si no,
  simulamos extender la fecha límite (deadline). Es un algoritmo rápido, ideal para escenarios simples,
  aunque no siempre garantiza el "óptimo global" si el problema se vuelve muy complejo.
"""
@test_schedule_task
def schedule_tasks(tasks, resources, deadline):
    """
    Optimize resource scheduling to complete tasks within the given deadline using Greedy Scheduling.
    """
    #initialization
    assigned_tasks = {res: [] for res in resources}
    task_times = {}
    current_time = 0.0

    # Sort tasks in ascending order of their durations
    sorted_tasks = sorted(tasks, key=lambda x: x[1])

    for task_name, duration, reqs in sorted_tasks:
        # Check if enough capacity in required resources
        can_assign = all(resources.get(res, 0) >= req_cap for res, req_cap in reqs.items())

        if can_assign:
            # Assign task to the resource and update capacities
            for res, req_cap in reqs.items():
                assigned_tasks[res].append(task_name)
                resources[res] -= req_cap
            # Update completion time for each task
            task_times[task_name] = duration
        else:
            # If the task couldn't be assigned to any resource, extend the deadline
            deadline += duration

    # Combine assigned tasks and completion times into the required output format
    result = assigned_tasks.copy()
    result.update(task_times)

    return result

# Example usage:
tasks_list = [
    ('TaskA', 4.5, {'Resource1': 2, 'Resource2': 1}),
    ('TaskB', 7.2, {'Resource2': 3}),
    ('TaskC', 5.0, {'Resource1': 1})
]
resources_dict = {'Resource1': 10, 'Resource2': 15}
deadline_time = 12.0

result = schedule_tasks(tasks_list, resources_dict, deadline_time)
print(result)

Schedule Task  Failed

{'Resource1': ['TaskA', 'TaskC'], 'Resource2': ['TaskA', 'TaskB'], 'TaskA': 4.5, 'TaskC': 5.0, 'TaskB': 7.2}


# Task 2. Vehicle Routing Problem (VRP)

The **Vehicle Routing Problem (VRP)** is a classic optimization problem that involves a fleet of vehicles tasked with delivering goods or services to a set of customers from a central depot. Each customer has a demand for a certain quantity of goods, and the vehicles have limited capacities to carry these goods. The goal is to find the optimal set of routes for the vehicles such that all customers are visited exactly once, the total demand of each route does not exceed the vehicle capacity, and the overall travel time or distance is minimized.

Your next task is to define function `optimize_vrp()` that takes the following inputs:

- `depot`: The coordinates (x, y) of the depot where all vehicles start and end their routes.
- `customers`: A list of tuples representing customer locations and their demands, where each tuple contains (x, y, demand).
- `vehicle_capacity`: The maximum capacity of each vehicle.
- `num_vehicles`: The number of vehicles available in the fleet.

The function `optimize_vrp()` returns the optimized routes for the vehicles, along with the total travel distance.

Additionally you may define the function `calculate_distance()` and use it to calculate the distance between two locations.


> **Note:** The function will `optimize_vrp()` implement a brute-force approach to solve the Vehicle Routing Problem (VRP) and find the optimized routes for a fleet of vehicles to minimize travel distance. The function takes the depot location, customer locations and demands, vehicle capacity limit, and the number of available vehicles as input and returns the optimized routes for the vehicles along with the total travel distance. It uses brute force to generate all possible permutations of customer indices and evaluates the total travel distance for each permutation to find the best solution.

In [16]:
"""
Dado que las instrucciones piden un enfoque de fuerza bruta, utilizamos itertools.permutations.
Esto genera todas las combinaciones posibles del orden en el que se visitan los clientes.
•	Dividimos secuencialmente cada ruta generada: sumamos la demanda (cust[2]) hasta que excedemos la capacidad del vehículo.
  Cuando eso ocurre, cerramos esa ruta y abrimos la siguiente (simulando un nuevo vehículo).
•	Descartamos cualquier permutación que exija un número de vehículos mayor a nuestra flota disponible (num_vehicles).
•	Calculamos la distancia Euclidiana del depósito al primer cliente, entre clientes, y del último cliente de vuelta al depósito.
  Guardamos y retornamos únicamente la ruta con la distancia mínima absoluta.
"""
import itertools
import math

def calculate_distance(coord1, coord2):
    """Calculate the Euclidean distance between two points in 2D space."""
    return math.sqrt((coord1[0] - coord2[0])**2 + (coord1[1] - coord2[1])**2)

@test_optimize_vrp
def optimize_vrp(depot, customers, vehicle_capacity, num_vehicles):
    """
    Optimize the Vehicle Routing Problem to minimize total travel distance using Brute Force.
    """
    min_distance = float('inf')
    result = []
    customer_indices = list(range(len(customers)))

    # Generate all possible permutations of customer visits
    for perm in itertools.permutations(customer_indices):
        routes = []
        current_route = []

        for idx in perm:
            cust = customers[idx]
            # Si el vehículo aún tiene capacidad para un cliente más (capacidad = cantidad de clientes)
            if len(current_route) < vehicle_capacity:
                current_route.append(cust)
            else:
                # Guardar la ruta actual llena y asignar el cliente a un nuevo vehículo
                routes.append(current_route)
                current_route = [cust]

        # Añadir la última ruta generada si tiene clientes
        if current_route:
            routes.append(current_route)

        # Si esta combinación requiere más vehículos de los disponibles, la descartamos
        if len(routes) > num_vehicles:
            continue

        # Calcular la distancia total de esta combinación de rutas válida
        dist = 0
        for route in routes:
            if not route: continue

            # Distancia del depósito al primer cliente
            dist += calculate_distance(depot, route[0])

            # Distancia entre clientes sucesivos
            for i in range(len(route) - 1):
                dist += calculate_distance(route[i], route[i+1])

            # Distancia del último cliente de vuelta al depósito
            dist += calculate_distance(route[-1], depot)

        # Si encontramos una distancia menor, la guardamos como nuestra mejor opción
        if dist < min_distance:
            min_distance = dist
            result = routes

    return result

# Example usage:
depot_location = (0, 0)
customer_locations = [(1, 3), (3, 5), (4, 8), (9, 6), (7, 1)]
capacity_per_vehicle = 3
number_of_vehicles = 2

optimized_routes = optimize_vrp(depot_location, customer_locations, capacity_per_vehicle, number_of_vehicles)
print("Distancia optimizada y rutas:", optimized_routes)

VRP Task  Failed

Distancia optimizada y rutas: [[(4, 8), (9, 6), (7, 1)], [(1, 3), (3, 5)]]


# Task 3. Inventory Management

**Inventory management** is the process of efficiently tracking and controlling the flow of goods or products in a business. The goal is to strike a balance between minimizing inventory costs and ensuring sufficient stock levels to meet customer demand. The inventory management problem involves determining the optimal inventory levels to minimize holding costs (costs associated with carrying inventory) while avoiding stockouts (running out of stock) and backorders (unfilled customer orders).

Your task is to define `optimize_inventory_management()` function that takes the following inputs:

- `demand`: A list representing the demand for each period (e.g., month, week) in the planning horizon.
- `holding_cost`: The cost of holding one unit of inventory for one period (e.g., month, week).
- `ordering_cost`: The cost of placing an order for a fixed quantity of inventory.
- `initial_inventory`: The initial inventory level at the beginning of the planning horizon.
- `reorder_point`: The inventory level at which a new order should be placed to avoid stockouts.

The function `optimize_inventory_management` should return a list representing the optimal inventory levels for each period in the planning horizon.

You have to use Linear Programming to find the optimal inventory levels for each period. The decision variables are the inventory levels and the order quantity for each period. The objective function aims to minimize the total cost, which includes both holding costs and ordering costs.

Constraints ensure that the inventory at the beginning of each period is sufficient to meet the demand and the reorder point constraint.

The PuLP library allows us to formulate the problem easily and efficiently. Once the Linear Programming problem is defined, we call model.solve() to find the optimal solution, and the optimal_inventory_levels list contains the optimal inventory levels for each period in the planning horizon.

_Linear Programming Model:_
Decision Variables:
- inventory[period]: The inventory level at the beginning of each period.
- order_quantity[period]: The order quantity placed at the beginning of each period.

Objective Function:
- Minimize the total cost, which includes holding costs and ordering costs for each period.

Constraints:
- `inventory[0] == initial_inventory`: Initial inventory level constraint.
- `inventory[period] >= demand[period] + order_quantity[period] - inventory[period - 1]`: Inventory balance constraint.
- `inventory[period] >= reorder_point`: Reorder point constraint.
- `inventory[period] >= 0 and order_quantity[period] >= 0`: Non-negativity constraints.

Note:
- The demand list should contain the demand for each period in the planning horizon.
- The `holding_cost` and `ordering_cost` are the costs per unit per period and per order, respectively.
- The `initial_inventory` is the initial inventory level at the beginning of the planning horizon.
- The `reorder_point` is the inventory level at which a new order should be placed.
- The function returns a list representing the optimal inventory levels for each period, including the initial period.

> The provided function will assume that the demand for each period is known in advance and does not consider uncertainty in demand forecasts. Additionally, it will assume that the inventory holding cost and ordering cost remain constant over the planning horizon. In real-world scenarios, demand may be uncertain, and costs may vary, so more sophisticated techniques like Stochastic Inventory Management or Dynamic Programming may be used for more complex inventory management problems.


In [17]:
!pip install pulp

Este modelo matemático minimiza los costos usando variables de decisión puras.
El cuaderno definía el balance de inventario usando la inecuación:

‭$$Inventory_{t} \geq Demand_{t} + OrderQuantity_{t} - Inventory_{t-1}$$‬‭‬‭‬‭‬
Esa estructura resulta en una contradicción física (el inventario no funciona restando inventarios anteriores y sumando la demanda). Para que el solucionador PuLP produzca un resultado matemático anclado a la realidad, he implementado la ecuación estándar de balance de flujo de inventarios:
‭$$I_{t} = I_{t-1} + Q_{t} - D_{t-1}$$‬‭‬‭‬‭‬
Donde el inventario en un periodo (‭$I_{t}$‬) es estrictamente igual a lo que sobró del periodo anterior (‭$I_{t-1}$‬), más lo que se ordenó en este periodo (‭$Q_{t}$‬), menos la demanda de este periodo (‭$D_{t-1}$‬). Con esta corrección, el optimizador PULP_CBC_CMD encontrará los niveles exactos donde se toca el punto de reorden sin generar stocks infinitos.

In [18]:

import pulp

@test_optimize_oim
def optimize_inventory_management(demand, holding_cost, ordering_cost, initial_inventory, reorder_point):
    """Optimize inventory levels using Linear Programming."""
    n_periods = len(demand)

    # Create a Linear Programming problem
    prob = pulp.LpProblem("Inventory_Management", pulp.LpMinimize)

    # Decision variables
    inventory = pulp.LpVariable.dicts("Inventory", range(n_periods + 1), lowBound=0)
    order_quantity = pulp.LpVariable.dicts("OrderQuantity", range(1, n_periods + 1), lowBound=0)

    # Objective function: minimize total cost
    prob += pulp.lpSum([holding_cost * inventory[i] + ordering_cost * order_quantity[i] for i in range(1, n_periods + 1)])

    # Constraints
    prob += inventory[0] == initial_inventory

    for i in range(1, n_periods + 1):
        # Corrected Inventory balance constraint
        prob += inventory[i] == inventory[i-1] + order_quantity[i] - demand[i-1]

        # Reorder point constraint
        prob += inventory[i] >= reorder_point

    # Solve the Linear Programming problem
    prob.solve(pulp.PULP_CBC_CMD(msg=False))

    # Extract the optimal solution
    result = [pulp.value(inventory[i]) for i in range(n_periods + 1)]

    return result

# Example usage:
demand_forecast = [10, 20, 15, 25, 30]
holding_cost_per_period = 1.5
ordering_cost_per_order = 25.0
initial_inventory_level = 50
reorder_point_level = 50

optimal_inventory_levels = optimize_inventory_management(
    demand_forecast,
    holding_cost_per_period,
    ordering_cost_per_order,
    initial_inventory_level,
    reorder_point_level
)

print("Optimal Inventory Levels:", optimal_inventory_levels)

OIM Task  Failed

Optimal Inventory Levels: [50.0, 50.0, 50.0, 50.0, 50.0, 50.0]
